In [ ]:
import pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    confusion_matrix,    ConfusionMatrixDisplay,    precision_score,    recall_score,    roc_auc_score,    roc_curve,    precision_recall_curve,)data = pd.read_csv("data.csv")print("Rows and columns:", data.shape)display(data.head())

In [ ]:
# Check for missing values and prepare the columns for the model.data = data.drop(columns=["id"], errors="ignore")data = data.dropna(axis=1, how="all")missing_values = data.isnull().sum()print("Missing values in each column:")print(missing_values[missing_values > 0] if missing_values.any() else "No missing values found")# Fill missing numbers with their column median and missing labels with the most common label.for column in data.select_dtypes(include="number").columns:    data[column] = data[column].fillna(data[column].median())data["diagnosis"] = data["diagnosis"].fillna(data["diagnosis"].mode()[0])# Plot missing-value counts and diagnosis counts.missing_to_plot = missing_values[missing_values > 0]if len(missing_to_plot) > 0:    missing_to_plot.plot(kind="bar", color="tomato", title="Missing values before filling")    plt.ylabel("Number of missing values")    plt.show()else:    print("All-empty columns were removed; no other missing values need filling.")data["diagnosis"].value_counts().reindex(["B", "M"]).plot(    kind="bar", color=["seagreen", "salmon"], title="Diagnosis counts")plt.xlabel("Diagnosis (B = benign, M = malignant)")plt.ylabel("Number of samples")plt.xticks(rotation=0)plt.show()# Convert diagnosis into numbers: benign = 0, malignant = 1.y = data["diagnosis"].map({"B": 0, "M": 1})X = data.drop(columns="diagnosis")

In [ ]:
# Split the data and standardize the features.X_train_validation, X_test, y_train_validation, y_test = train_test_split(    X, y, test_size=0.20, random_state=42, stratify=y)X_train, X_validation, y_train, y_validation = train_test_split(    X_train_validation, y_train_validation,    test_size=0.20, random_state=42, stratify=y_train_validation)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_validation_scaled = scaler.transform(X_validation)X_test_scaled = scaler.transform(X_test)print("Training examples:", len(y_train))print("Validation examples:", len(y_validation))print("Test examples:", len(y_test))

In [ ]:
# Train logistic regression and check its predictions on the test data.model = LogisticRegression(max_iter=1000, random_state=42)model.fit(X_train_scaled, y_train)test_probability = model.predict_proba(X_test_scaled)[:, 1]test_prediction = (test_probability >= 0.50).astype(int)print("Confusion matrix at threshold 0.50:")print(confusion_matrix(y_test, test_prediction))print("Precision:", round(precision_score(y_test, test_prediction), 3))print("Recall:", round(recall_score(y_test, test_prediction), 3))print("ROC-AUC:", round(roc_auc_score(y_test, test_probability), 3))# Plot the ROC curve. A curve closer to the top-left is generally better.false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, test_probability)plt.plot(false_positive_rate, true_positive_rate, label="Logistic Regression")plt.plot([0, 1], [0, 1], "--", color="gray", label="Random guess")plt.xlabel("False Positive Rate")plt.ylabel("True Positive Rate (Recall)")plt.title("ROC Curve")plt.legend()plt.grid(True)plt.show()

In [ ]:
# Choose a threshold using validation data, then check it on the test data.validation_probability = model.predict_proba(X_validation_scaled)[:, 1]validation_precision, validation_recall, possible_thresholds = precision_recall_curve(    y_validation, validation_probability)f1_scores = (    2 * validation_precision[:-1] * validation_recall[:-1]    / (validation_precision[:-1] + validation_recall[:-1] + 1e-10))best_threshold = possible_thresholds[f1_scores.argmax()]print("Chosen threshold:", round(best_threshold, 3))# A higher threshold usually gives fewer positive predictions; it can change precision and recall.tuned_prediction = (test_probability >= best_threshold).astype(int)print("Tuned precision:", round(precision_score(y_test, tuned_prediction), 3))print("Tuned recall:", round(recall_score(y_test, tuned_prediction), 3))ConfusionMatrixDisplay.from_predictions(    y_test, tuned_prediction, display_labels=["Benign", "Malignant"], cmap="Blues")plt.title("Confusion Matrix at Tuned Threshold")plt.show()# Sigmoid changes the model's score into a probability between 0 and 1.print("Sigmoid formula: probability = 1 / (1 + exp(-score))")print("The threshold decides when that probability is classified as malignant.")